# 08 · INT8 calibration ablation: one factor at a time (24 engines)

**QPAD — Calibration-Set Composition Bias in INT8 Face Presentation Attack Detection**
CS 228 Biometric Security with AI · San José State University · Ananya Tyagi & Vindhya Channarayapatna Ravishankar

**Priority:** 1 (follow-up to the pilot review)

**Purpose.** Starting from the natural calibration set (600 images, 67% attack images), change one factor at a time and build each setting with 3 seeds: attack share 67/40/20/10/0% at 600 images, and 600/200/60/20 images.

**In plain words.** The pilot found almost no difference between calibration mixes, and the professor's review traced most of the spread to the per-engine threshold. Here we push one factor much further than the pilot did, to find where results start to get worse. Each setting is built 3 times with different random video picks, so we can compare ranges instead of single engines. This notebook only builds and scores engines; 08b analyses them.

**Expected inputs**
- Passed `equivalence.json`; ONNX model; processed train calibration frames; dev/test evaluation frames; FP32 `threshold.json`. Same GPU type and TensorRT version as the pilot (notebook 07).

**Expected outputs**
- `QPAD_Project/results/int8/ablation/calibration_manifest.csv` (saved before any engine)
- `QPAD_Project/engines/ablation/int8_entropy_<setting>_s<seed>/` (engine, cache, scales, metadata)
- `QPAD_Project/results/int8/ablation/int8_entropy_<setting>_s<seed>/` (dev/test scores, metrics)
- `QPAD_Project/results/int8/ablation/engines.csv`

**Safety rules.** The OULU-NPU dataset is access-restricted. This notebook never uploads data, never creates
share links, never prints credentials, and never writes videos or face crops to the Git repository.
Long operations run only after you change `DRY_RUN = True` to `False` and read the pre-flight summary.

In [ ]:
# --- Step 1: mount Google Drive (private storage) -------------------------------------------
# Authentication happens in Colab's own pop-up; nothing about it is stored in this notebook.
import os, sys, json, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not os.environ.get("QPAD_SKIP_DRIVE_MOUNT"):
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Drive mount skipped (not in Colab, or test mode).")

In [ ]:
# --- Step 2: make the QPAD code importable ------------------------------------------------------
# Option A (default): upload the whole `qpad/` repository folder to
#   MyDrive/QPAD_Project/code/qpad
# Option B: set REPO_URL to your PRIVATE GitHub repository; Colab/GitHub will ask you to authenticate.
#   Never paste access tokens into this notebook.
REPO_URL = ""  # e.g. "https://github.com/<account>/qpad.git"
QPAD_CODE_DIR = Path(os.environ.get("QPAD_CODE_DIR", "/content/drive/MyDrive/QPAD_Project/code/qpad"))
if REPO_URL:
    QPAD_CODE_DIR = Path("/content/qpad")
    if not (QPAD_CODE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(QPAD_CODE_DIR)], check=True)
if not (QPAD_CODE_DIR / "src" / "qpad").is_dir():
    raise FileNotFoundError(f"QPAD code not found at {QPAD_CODE_DIR}. Upload the repo folder or set REPO_URL.")
sys.path.insert(0, str(QPAD_CODE_DIR / "src"))

if IN_COLAB and not os.environ.get("QPAD_SKIP_PIP"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(QPAD_CODE_DIR / "requirements-colab.txt")], check=True)
import qpad
print("qpad", qpad.__version__, "from", QPAD_CODE_DIR)

In [ ]:
# --- Step 3: authoritative paths -------------------------------------------------------------------
# These are the ONLY data/output locations. pathlib handles any spaces in paths.
from qpad.config import ProjectPaths, load_config, get_logger, set_global_seed, disk_free_gb

DATASET_ROOT = Path("/content/drive/MyDrive/QPAD_Project/OULU_NPU/raw")
TRAIN_TAR = DATASET_ROOT / "Train_files-002.tar"
DEV_TAR = DATASET_ROOT / "Dev_files-001.tar"
TEST_TAR = DATASET_ROOT / "Test_files-003.tar"
METADATA_ZIP = DATASET_ROOT / "OULU-NPU-20261010T032906Z-1-004.zip"
PROJECT_DRIVE_ROOT = Path("/content/drive/MyDrive/QPAD_Project")
LOCAL_WORK_ROOT = Path("/content/qpad_work")

cfgs = load_config(QPAD_CODE_DIR / "configs")
paths = ProjectPaths.from_config(cfgs["dataset"])
if not paths.overrides and not cfgs["_overrides"]:  # YAML must agree with the constants above
    assert paths.dataset_root == DATASET_ROOT and paths.project_drive_root == PROJECT_DRIVE_ROOT
    assert paths.archive("train") == TRAIN_TAR and paths.archive("dev") == DEV_TAR
    assert paths.archive("test") == TEST_TAR and paths.archive("metadata") == METADATA_ZIP
    assert paths.local_work_root == LOCAL_WORK_ROOT
for o in cfgs["_overrides"]:
    print("WARNING (test mode):", o)
paths.ensure_output_dirs()
print(paths.describe())

In [ ]:
# --- Step 4: software, GPU and disk information ------------------------------------------------
from qpad.reproducibility import environment_info, git_commit
from qpad.nbutils import df_h, tqdm_progress, preflight

ENV_INFO = environment_info()
print(json.dumps(ENV_INFO, indent=1, default=str))
print("git:", git_commit(QPAD_CODE_DIR))
print(df_h(LOCAL_WORK_ROOT if LOCAL_WORK_ROOT.exists() else Path("/content") if Path("/content").exists() else Path.cwd()))

In [ ]:
# --- Step 5: run configuration and seeds ---------------------------------------------------------
DRY_RUN = True                 # change to False ONLY after the dry run looks correct
DRY_RUN_VIDEOS_PER_CLASS = 2   # per split and class, deterministic (first IDs)
FORCE_REBUILD = False          # True re-creates outputs that already exist
RETRY_FAILED = False           # True re-processes items recorded as failed
RANDOM_SEED = 42
DRY_RUN_EVAL_VIDEOS_PER_CLASS = 3   # dry run: one engine (natural, seed 1), evaluated on 3 videos/class/split
ALLOW_OTHER_GPU = False   # True: run the whole ablation on a different GPU than the pilot (e.g. L4 instead of A100)
if os.environ.get("QPAD_FORCE_DRY_RUN"):
    DRY_RUN = True
SEED_INFO = set_global_seed(RANDOM_SEED)
logger = get_logger("qpad.nb08", paths.logs / "nb08.log")
logger.info("DRY_RUN=%s seed=%s", DRY_RUN, RANDOM_SEED)
from qpad.pipeline import output_paths
OUT = output_paths(paths, DRY_RUN)
OUT.ensure_output_dirs()
print("Outputs of this run go to:", OUT.project_drive_root)

In [ ]:
# TensorRT is needed from here on. Its version is pinned in configs/int8_experiments.yaml: TensorRT 11
# removed the INT8 entropy calibrator that the pre-registered method uses. Colab preinstalls TensorRT 11,
# so the first time this cell swaps it for the pinned version and asks for ONE session restart.
TRT_PIN = str(cfgs["int8"]["tensorrt"]["version"])
def installed_trt():   # checked in a fresh process: an already-imported module would hide a reinstall
    r = subprocess.run([sys.executable, "-c", "import tensorrt; print(tensorrt.__version__)"],
                       capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else None
if IN_COLAB and installed_trt() != TRT_PIN:
    import torch
    cuda_major = torch.version.cuda.split(".")[0]
    old = [p["name"] for p in json.loads(subprocess.run([sys.executable, "-m", "pip", "list", "--format=json"],
                                                        capture_output=True, text=True, check=True).stdout)
           if p["name"].lower().replace("_", "-").startswith("tensorrt")]
    if old:
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", *old], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"tensorrt-cu{cuda_major}=={TRT_PIN}"], check=True)
    if installed_trt() != TRT_PIN:
        raise RuntimeError(f"installing TensorRT {TRT_PIN} failed (found {installed_trt()})")
    if "tensorrt" in sys.modules:
        raise RuntimeError(f"TensorRT {TRT_PIN} is now installed. Runtime -> Restart session, then Run all again "
                           "(needed once per runtime; the old version is still loaded in this session).")
try:
    import tensorrt as trt
except ImportError:
    trt = None
print("TensorRT:", getattr(trt, "__version__", None))
if trt is not None and trt.__version__ != TRT_PIN:
    raise RuntimeError(f"TensorRT {trt.__version__} is loaded but {TRT_PIN} is pinned: restart the session.")
from qpad.tensorrt_utils import hardware_fingerprint
HW = hardware_fingerprint(); print(HW)

In [ ]:
import json, torch, pandas as pd
from qpad.ablation import settings_table
from qpad.experiments import engine_tag
from qpad.pipeline import find_input
if not torch.cuda.is_available():
    raise RuntimeError("INT8 TensorRT engines need a CUDA GPU runtime.")
icfg, acfg = cfgs["int8"], cfgs["int8"]["ablation"]
METHOD = icfg["calibration_method"]["primary"]   # pre-registered: entropy
BATCH = int(icfg["calibration_method"]["calibration_batch_size"])
SETTINGS_TABLE = settings_table(acfg, int(icfg["calibration_budget"]["frames_per_video"]), BATCH)
display(SETTINGS_TABLE)
SEEDS = acfg["seeds"][:1] if DRY_RUN else list(acfg["seeds"])
SETTINGS = [acfg["baseline"]] if DRY_RUN else list(SETTINGS_TABLE["setting"])
print("method:", METHOD, "| seeds:", SEEDS, "| settings:", SETTINGS, "| engines:", len(SEEDS) * len(SETTINGS))
# Engines are only comparable on the same GPU type and TensorRT version (kernel choice differs between GPUs).
pilot_meta = find_input(paths, DRY_RUN, "engines", engine_tag(METHOD, "natural", 1)) / "engine_meta.json"
if pilot_meta.is_file():
    pilot_hw = json.loads(pilot_meta.read_text())["hardware"]
    if (pilot_hw["gpu_name"], pilot_hw["tensorrt"]) != (HW["gpu_name"], HW["tensorrt"]):
        msg = (f"Pilot engines were built on {pilot_hw['gpu_name']} with TensorRT {pilot_hw['tensorrt']}, "
               f"this runtime has {HW['gpu_name']} with TensorRT {HW['tensorrt']}.")
        if not ALLOW_OTHER_GPU:
            raise RuntimeError(msg + " Use the same GPU type, or set ALLOW_OTHER_GPU = True to build the whole "
                               "ablation on this one.")
        print("WARNING:", msg, "The ablation settings are compared with each other, not with the pilot engines.")
else:
    print("pilot engine metadata not found; GPU type not compared")
# Every ablation engine must come from one GPU type and TensorRT version.
built = {(m["hardware"]["gpu_name"], m["hardware"]["tensorrt"])
         for m in (json.loads(p.read_text()) for p in (OUT.engines / "ablation").glob("*/engine_meta.json"))}
if built - {(HW["gpu_name"], HW["tensorrt"])}:
    raise RuntimeError(f"Ablation engines already exist from {sorted(built)}; finish the run on that GPU type.")

In [ ]:
from qpad.experiments import build_ablation_manifest
CAL, CAL_STORE, CAL_PATH = build_ablation_manifest(paths, cfgs, DRY_RUN, SEEDS, SETTINGS)
print("calibration manifest saved BEFORE building engines:", CAL_PATH)
counts = CAL.groupby(["seed", "condition", "class"]).size().unstack(fill_value=0)
counts["images"] = counts.sum(axis=1)
display(counts)

In [ ]:
from qpad.data_exceptions import expected_eval_counts
from qpad.datasets import materialize
from qpad.experiments import balanced_subset, build_int8_engine, calibration_array, engine_is_complete, evaluate_backend
from qpad.pipeline import processed_split_dir
from qpad.tensorrt_utils import TrtRunner
norm = cfgs["preprocessing"]["normalization"]
nfr = expected_eval_counts(cfgs)
n_eval = DRY_RUN_EVAL_VIDEOS_PER_CLASS if DRY_RUN else None
DEV = balanced_subset(materialize(processed_split_dir(paths, DRY_RUN, "dev"), "dev", "eval", OUT.local_cache), n_eval)
TEST = balanced_subset(materialize(processed_split_dir(paths, DRY_RUN, "test"), "test", "eval", OUT.local_cache), n_eval)
FP32_THR = json.loads(find_input(paths, DRY_RUN, "results", "fp32/threshold.json").read_text())["threshold"]
ABL_ENGINES, ABL_RESULTS = OUT.engines / "ablation", OUT.results / "int8" / "ablation"
preflight("INT8 ablation engines", len(SEEDS) * len(SETTINGS), [OUT.models / "deeppixbis_fp32.onnx", CAL_PATH],
          [ABL_ENGINES, ABL_RESULTS], LOCAL_WORK_ROOT, {"method": METHOD, **icfg["tensorrt"]}, SEEDS, DRY_RUN,
          "~4-7 min per engine (build + scoring); finished engines are skipped if the session restarts")

## Build and score the engines
Seed by seed, so an interrupted session still leaves complete seeds. Finished engines are verified by hash and skipped on a rerun. Only policy A (fixed FP32 threshold) is printed here; 08b adds the other rules.

In [ ]:
for seed in SEEDS:
    for setting in SETTINGS:
        tag = f"ablation/{engine_tag(METHOD, setting, seed)}"
        edir, rdir = OUT.engines / tag, OUT.results / "int8" / tag
        if engine_is_complete(edir, rdir) and not FORCE_REBUILD:
            print("verified, skipping", tag)
        else:
            images = calibration_array(CAL_STORE, CAL, seed, setting, norm)
            build_int8_engine(paths, cfgs, DRY_RUN, images, tag, METHOD)
            evaluate_backend(TrtRunner(edir / "engine.plan"), DEV, TEST, norm, FP32_THR, rdir, nfr,
                             int(icfg["tensorrt"]["opt_batch"]))
        m = json.loads((rdir / "metrics.json").read_text())["policy_A"]
        print(tag, "| A: print", m["print_apcer"]["text"], "| replay", m["replay_apcer"]["text"], "| BPCER", m["bpcer"]["text"])

## Did every engine see its whole calibration set?
Each engine should have received images / 20 batches and written one scale per calibrated tensor.

In [ ]:
rows = []
for seed in SEEDS:
    for setting in SETTINGS:
        tag = f"ablation/{engine_tag(METHOD, setting, seed)}"
        meta = json.loads((OUT.engines / tag / "engine_meta.json").read_text())
        images = int(meta["n_calibration_images"])
        rows.append({"setting": setting, "seed": seed, "images": images, "batches_served": meta["calibration_batches_served"],
                     "expected_batches": images // BATCH, "scaled_tensors": meta["n_scaled_tensors"],
                     "gpu": meta["hardware"]["gpu_name"], "tensorrt": meta["hardware"]["tensorrt"],
                     "sha256": meta["sha256"][:12]})
ENGINES = pd.DataFrame(rows)
ENGINES["ok"] = (ENGINES["batches_served"] == ENGINES["expected_batches"]) & \
                (ENGINES["scaled_tensors"] == ENGINES["scaled_tensors"].mode().iloc[0])
ENGINES.to_csv(ABL_RESULTS / "engines.csv", index=False)
display(ENGINES)
print("engines with their full calibration set:", f"{int(ENGINES['ok'].sum())}/{len(ENGINES)}")

**Next notebook:** `08b_int8_ablation_analysis.ipynb`